# Transfer Learning

Most image recognition tasks don't have enough data to adequately train a full-sized convolutional neural network.  As a result, it's common to instead do *transfer learning*.  In transfer learning, we first train a large neural net on a very large dataset we do have (or, more likely, we let someone else do it for us).  Presumably, those convolutional layers will then be able to extract useful features for all sorts of image tasks, not just the ones they were specifically trained for.

We then take that large, trained neural net, chop off the fully-connected part at the end, replace it with a randomly-initialized new fully-connected part, and then train on our smaller dataset, only allowing those new layers to train.  Given the good features from the convolutional layers, we can hopefully build a good network for our specific task.

You're going to build a neural net which performs classification between two types of things of your choice using transfer learning.  To do this, we're going to scrape Bing Images (yes! Bing!) for our dataset of images.

[Here is your Gem](https://gemini.google.com/gem/1aAq9DuXbhJjpuH1JdpFU8WdNeBj10XTi?usp=sharing).

`~/bin/submit -c=SD312 -p=lab09_transfer_learning index.ipynb`

## Step 0: Set up your environment

This project has a couple extra requirements, so make a new virtual environment.

- `mamba create -n transfer numpy scipy scikit-learn pandas plotly matplotlib jupyter opencv imutils tqdm torchinfo`
- `mamba activate transfer`
- `pip install torch torchvision standard-imghdr`
- Make and cd to a personal folder in `/SD312` for you and this lab
- `mkdir imgs && cd imgs`
- `pip install git+https://github.com/ostrolucky/Bulk-Bing-Image-downloader`
- `cp ~/.local/bin/bbid* .`

If that last command doesn't work, try `cp ~/miniforge3/envs/transfer/bin/bbid* .` instead.

Then, in order to get around ITSD's nonsense, disable SSL verification by
opening up `bbid.py`, and just below all the import statements, add:

```python
import ssl
_create_unverified_https_context = ssl._create_unverified_context
ssl._create_default_https_context = _create_unverified_https_context
```

## Step 1: Build a dataset

You can download images of whatever you like to build your classifier.  Inside `~/imgs` is a file called `bbid.py`.  If you run `python bbid.py goats`, it will download a bunch of pictures of goats into `imgs/bing`.  You can then use [this script](mvr.sh) as `bash mvr.sh goats` to move most of them into `imgs/train/goats`, and the rest into `imgs/test/goats`.

You'll want to do this for at least two types of things, of your choice, to build a classifier between.

## Step 2: Train a neural net, based off the ResNet18 architecture

Understand the code in this notebook, and finish the training and testing loop. 

This code has a few extra bells and whistles, so read through to understand.

First, imports and "let's do insecure things to access the internet, because ITSD has a MITM attack."

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets,transforms
import torchinfo
from torch.utils.data import DataLoader
from torchvision.transforms import v2

import ssl
_create_unverified_https_context = ssl._create_unverified_context
ssl._create_default_https_context = _create_unverified_https_context

Our datasets will be built by pointing at the directories that contain the training and testing data. The different classes must be split within these folders. For example, `/SD312/<youralpha>/lab9/imgs/train` and `/SD312/<youralpha>/lab9/imgs/test` might have two folders each, called `goats` and `mules`.

In [ ]:
#locations of training and testing data
train_data_path= # full path of your training data, ending with /imgs/train
test_data_path= # full path of your testing data, ending with /imgs/test

To reduce overfitting, it's common to randomly transform each training image so that it never shows up in quite the same way twice. We do not want to do that for our testing images, but we do need to resize them to the expected size and convert them to Tensors.

We build our `Dataset` and `DataLoader` objects to read from the folders and apply the given transformations.

In [ ]:
# Transformations for training: Ensure size matches model expectations (usually 224 for ResNet)
random_transform = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=30),
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transformations for testing: Ensure size matches model expectations (usually 224 for ResNet)
common_transform = v2.Compose([
    v2.Resize(size=(224, 224), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#build datasets and dataloaders
train_dataset=datasets.ImageFolder(root=train_data_path,transform=random_transform)
test_dataset=datasets.ImageFolder(root=test_data_path,transform=common_transform)
batch_size=32

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True,
    persistent_workers=True
)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

We'll start with ResNet18, which is a relatively small image classification network with about 11 million parameters. Have a look at the construction of the network, paying attention to both the feature extraction portion, and the final layers.

In [ ]:
#get the model and its weights
model=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
#print out the summary of it. What's its final output size? Why?
print(model)

To do transfer learning, we tell the network not to train the feature extractor. We set `.requires_grad` for all these given parameters to `False` so they don't train.

We then overwrite `model.fc` with one of our own. This will have random parameters, and will need to have the output format for our task.

In addition, we push the model and its parameters to the GPU.

In [ ]:
#Set the model parameters to be non-changeable
for param in model.parameters():
    param.requires_grad=False

# Identify the number of input features to the existing head
num_ftrs = model.fc.in_features

# Dynamically set output classes based on the dataset
num_classes = len(train_dataset.classes)

# Replace the head
model.fc = nn.Linear(num_ftrs, num_classes)
model = model.to('cuda')

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; color: #856404;">
  <strong>🟡 AI Policy: YELLOW</strong> <br>
  Generative AI is allowed, with limitations.
</div>

Now that you have your model, build your training and testing loops. Run it on your dataset, and talk about how it does.

<div style="background-color: #d4edda; border-left: 6px solid #28a745; padding: 15px; color: #155724;">
  <strong>🟢 AI Policy: GREEN</strong> <br>
  Generative AI is allowed/encouraged for this section.
</div>

## Step 3: Play

Look for things to modify.  Depth?  Width?  Dropout layers (google it!)? Activation functions?  More classes of images? Number of epochs?

Here are some questions you can explore:

- How does adding more classes impact the performance of your learned model? Are some classes harder to differentiate from each other than others (use a confusion matrix)?
- How does adding more layers impact the speed of convergence and the ultimate performance of the model?
- Can you find some classes that you are unable to learn particularly well?
- Does changing the learning rate have any effect?
- If you remove the random transformations to the training set, or add others, how does test performance change?  Does it take longer to overfit?
- If you don't train long enough, you'll underfit. If you train too long, you'll overfit. Can you implement early stopping (like many did for the movie recommendation project) so you stop at just the right time?
- There are [many possible backbones you can use](https://docs.pytorch.org/vision/main/models.html#classification). Try some. Explore tradeoffs between accuracy and speed.